In [1]:
# Cell 1 — Read
from pyspark.sql import functions as F
from pyspark.sql import types as T

inv_df = spark.read \
    .option("header", "true") \
    .csv("Files/raw/inventory/inventory_snapshot.csv")

print("Raw inventory:", inv_df.count())


StatementMeta(, b7bd11ce-57f6-4ab3-afea-7b8673212361, 3, Finished, Available, Finished, False)

Raw inventory: 610


In [4]:
# Cell 2 — Clean and write
cleaned_inv = inv_df \
    .withColumn("quantity_available", F.col("quantity_available").cast(T.IntegerType())) \
    .withColumn("quantity_reserved",  F.col("quantity_reserved").cast(T.IntegerType())) \
    .withColumn("reorder_threshold",  F.col("reorder_threshold").cast(T.IntegerType())) \
    .withColumn("last_updated",       F.to_timestamp(F.col("last_updated"))) \
    .withColumn(
        # Flag items below reorder level
        "needs_reorder",
        F.col("quantity_available") <= F.col("reorder_threshold")
    ) \
    .withColumn("silver_created_at", F.current_timestamp()) \
    .filter(F.col("product_id").isNotNull())

# Silver Lakehouse Delta path
TARGET_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables/silver_inventory"
)

# Write as Delta
(
    cleaned_inv.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(TARGET_PATH)
)


# Validate
silver_inventory = (
    spark.read
    .format("delta")
    .load(TARGET_PATH)
)

row_count = silver_inventory.count()

print("=" * 60)
print("✅ Silver Inventory table created successfully!")
print(f"📂 Path      : {TARGET_PATH}")
print(f"📊 Total Rows: {row_count}")
print("=" * 60)

silver_inventory.show(10, truncate=False)

StatementMeta(, b7bd11ce-57f6-4ab3-afea-7b8673212361, 6, Finished, Available, Finished, False)

✅ Silver Inventory table created successfully!
📂 Path      : abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/silver_lakehouse.Lakehouse/Tables/silver_inventory
📊 Total Rows: 610
+----------+--------------+------------------+-----------------+-----------------+-------------------+-------------+--------------------------+
|product_id|warehouse_code|quantity_available|quantity_reserved|reorder_threshold|last_updated       |needs_reorder|silver_created_at         |
+----------+--------------+------------------+-----------------+-----------------+-------------------+-------------+--------------------------+
|PROD1000  |WH_MUM_01     |155               |35               |50               |2026-07-01 18:21:04|false        |2026-07-02 16:55:59.664637|
|PROD1000  |WH_DEL_01     |300               |45               |20               |2026-07-01 18:21:04|false        |2026-07-02 16:55:59.664637|
|PROD1000  |WH_HYD_01     |91                |3                |10               |20

In [5]:
from pyspark.sql import functions as F

# Base ABFS path for Silver Lakehouse Tables
BASE_PATH = (
    "abfss://RetailPulse_DataPlatform@onelake.dfs.fabric.microsoft.com/"
    "silver_lakehouse.Lakehouse/Tables"
)

tables = [
    "silver_orders",
    "silver_customers",
    "silver_products",
    "silver_inventory"
]

print("SILVER LAYER SUMMARY")
print("=" * 50)

for table in tables:
    table_path = f"{BASE_PATH}/{table}"
    
    row_count = (
        spark.read
        .format("delta")
        .load(table_path)
        .count()
    )
    
    print(f"{table:<25} {row_count:>8} rows")

print("=" * 50)
print("✅ All Silver tables validated successfully!")

StatementMeta(, b7bd11ce-57f6-4ab3-afea-7b8673212361, 7, Finished, Available, Finished, False)

SILVER LAYER SUMMARY
silver_orders                 1400 rows
silver_customers               500 rows
silver_products                200 rows
silver_inventory               610 rows
✅ All Silver tables validated successfully!
